# TideVec quickstart: retrieval that knows when facts change

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishodu2023/TideVec/blob/main/examples/notebooks/tidevec_quickstart.ipynb)

This notebook runs TideVec **in-process** (no server) with a real sentence-transformers model and shows:

1. Temporal scoring inside the ANN search: fresh documents win ties against stale ones.
2. Supersession: a revised document is linked to the version it replaces, automatically.
3. Using it as a LangChain retriever.

In [ ]:
# The embedded engine compiles from source (about 1-2 minutes on Colab).
%pip install -q "tidevec[langchain]" sentence-transformers \
    "git+https://github.com/ashishodu2023/TideVec#subdirectory=sdk/python-embedded"

In [ ]:
import time
from tidevec import Local, HalfLife
from tidevec.embeddings import SentenceTransformerEmbedder

emb = SentenceTransformerEmbedder("sentence-transformers/all-MiniLM-L6-v2")
db = Local("./tidevec_data")
DAY = 86_400_000
now = int(time.time() * 1000)

if db.has_collection("policies"):
    db.drop_collection("policies")
db.create_collection("policies", dim=emb.dim, half_life_ms=HalfLife.ONE_MONTH,
                     temporal_blend=0.3, supersession=True,
                     supersede_threshold=0.8, supersede_match_field="doc_id")

In [ ]:
docs = [
    ("refunds-v1", "Refunds are issued within 14 days of the return request.", "refunds", 200),
    ("shipping-v1", "Standard shipping takes 5 business days.", "shipping", 150),
    ("refunds-v2", "Refunds are now issued within 3 days of the return request.", "refunds", 2),
]
for vid, text, doc_id, age in docs:
    db.upsert("policies", [{"id": vid, "embedding": emb.embed_query(text),
                            "created_at": now - age * DAY,
                            "payload": {"text": text, "doc_id": doc_id}}])

q = emb.embed_query("How long do refunds take?")
for title, kwargs in [("pure similarity", {"temporal_blend": 0.0}),
                      ("TideVec default", {}),
                      ("superseded pushed down", {"superseded_penalty": 1.0})]:
    print(f"\n{title}")
    for h in db.search("policies", q, top_k=3, **kwargs):
        print(f"  {h.score:.3f} {h.id:<12} superseded_by={h.superseded_by}")

## As a LangChain retriever

In [ ]:
from tidevec.integrations.langchain import TideVecVectorStore

store = TideVecVectorStore(db, "handbook", emb, supersession=True,
                           supersede_threshold=0.8, supersede_match_field="doc_id")
store.add_texts(["Employees may work remotely 2 days per week."],
                metadatas=[{"doc_id": "remote"}], created_at=now - 300 * DAY)
store.add_texts(["Employees may now work remotely 4 days per week."],
                metadatas=[{"doc_id": "remote"}])

retriever = store.as_retriever(search_kwargs={"k": 2, "exclude_superseded": True})
[d.page_content for d in retriever.invoke("How many remote days are allowed?")]